# Chat Agent

### State Diagram (Agent View)
```mermaid
stateDiagram-v2
direction LR
INIT --> CHAT
CHAT --> FINAL
```


### a) Create Agent

In [1]:
from gai.asm.agents import ChatAgent
from gai.mcp.client.mcp_client import McpAggregatedClient
from gai.lib.config import config_helper

from gai.lib.tests import make_local_tmp
import os
here = make_local_tmp()
file_path = os.path.join(here, "monologue.json")
from gai.messages import FileMonologue
monologue = FileMonologue(agent_name="ChatAgent",file_path=file_path)

# Create an artificial dialogue history for testing
from gai.messages import FileDialogue, MessagePydantic
messages = [
    MessagePydantic(**{
        'id': 'b1e5f98c-f6eb-47de-a6e2-387510d970f9', 
        'header': {
            'sender': 'User',
            'recipient': 'Sara',
            "timestamp": 1751308157.270983,
            "order": 0            
        }, 'body': {
            'type': 'chat.send',
            'dialogue_id': '00000000-0000-0000-0000-000000000000',
            'round_no': 0,
            'step_no': 0,
            'role': "user",
            'content': 'I love horror stories, are you familiar with them?',
        }
    }),
    MessagePydantic(**{
        'id': 'abbc7961-45dc-4973-aaf4-a6224ed35d37', 
        'header': {
            'sender': 'Sara',
            'recipient': 'User',
            "timestamp": 1751308167.3488164,
            "order": 1
        }, 'body': {
            'type': 'chat.reply',
            'dialogue_id': '00000000-0000-0000-0000-000000000000',
            'round_no': 0, 
            'step_no': 1,
            'chunk_no':10,
            'chunk':'<eom>',
            'role': "assistant",
            'content': 'Yes, I am familiar with horror stories. They are a fascinating genre that can evoke strong emotions and create a sense of suspense and fear. Do you have any specific horror stories in mind that you would like to discuss?'
        }
    })]
from gai.lib.constants import DEFAULT_GUID
file_path = os.path.join(here, f"{DEFAULT_GUID}.json")
dialogue = FileDialogue(messages=messages,file_path=file_path)
recap = dialogue.extract_recap()

aggregated_client = McpAggregatedClient(["mcp-pseudo","mcp-time", "mcp-web"])
tools = await aggregated_client.list_tools()
agent = ChatAgent(
    agent_name="ChatAgent",
    llm_config=config_helper.get_client_config(
        {
            "client_type": "anthropic",
            "model": "claude-sonnet-4-20250514",
            "extra": {
                "max_tokens": 32000,
                "temperature": 0.7,
                "top_p": 0.95,
                "tools": True,
                "stream": True,
            },
        }
    ),
    monologue=monologue,
    recap=recap
)

### reset monologue (optional)

In [2]:
monologue.reset()


### b) run_async

In [3]:
user_message="Tell me a one paragraph story."
resp=await agent.run_async(user_message=user_message)
# Stream the response
async for chunk in resp():
    if chunk:
        if isinstance(chunk, str):
            print(chunk, end="", flush=True)

Here's a chilling tale for you: The old music box had been silent for decades, tucked away in the dusty corner of Margaret's inherited attic, until the night she wound its tiny silver key out of curiosity. As the haunting melody began to play, she watched in fascination as the delicate porcelain ballerina spun gracefully—but her blood ran cold when she realized the dancer's painted face was slowly turning to look directly at her, its tiny mouth opening wider with each rotation, revealing rows of needle-sharp teeth that glinted in the moonlight streaming through the window. The music grew faster, more frantic, and Margaret found herself unable to move, unable to look away, as the ballerina's pirouettes became increasingly violent, spinning so rapidly that the music box began to shake and smoke, while somewhere in the distance, she could hear the sound of children laughing—the same children who had mysteriously vanished from the neighborhood fifty years ago, the same week the music box's

### c) Show monologue

In [4]:
import json

# Show the monologue
print("\n───────────────────────── MONOLOGUE START ─────────────────────────")
messages = agent.fsm.monologue.list_messages()
for message in messages[-2:]:
    print(json.dumps(message.model_dump(), indent=4))
print("───────────────────────── MONOLOGUE END ─────────────────────────\n")

# Print memory size
mem_size = agent.fsm.monologue.get_total_size()
print("Total char size=", mem_size)


───────────────────────── MONOLOGUE START ─────────────────────────
{
    "id": "17dfcc14-470a-44d5-811c-214ce151585b",
    "header": {
        "sender": "User",
        "recipient": "ChatAgent",
        "timestamp": 1752480981.0166516,
        "order": 0
    },
    "body": {
        "type": "monologue",
        "state_name": "CHAT",
        "step_no": 1,
        "content_type": "text",
        "role": "user",
        "content": "\n            Here is a recap of the conversation:\n            User: I love horror stories, are you familiar with them?\nSara: Yes, I am familiar with horror stories. They are a fascinating genre that can evoke strong emotions and create a sense of suspense and fear. Do you have any specific horror stories in mind that you would like to discuss?\n            \n            You may respond to my following message using the context you have learnt.\n            Tell me a one paragraph story.\n            "
    }
}
{
    "id": "a875630d-721c-4dfa-98da-eb012e616d

### d) Show dialogue

In [5]:
dialogue.add_user_message(recipient='Sara', content=user_message)
dialogue.add_assistant_message(sender='Sara', chunk="<eom>", content=agent.final_output())
for msg in dialogue.list_messages():
    print(f"{msg.header.sender}: {msg.body.content}")

User: I love horror stories, are you familiar with them?
Sara: Yes, I am familiar with horror stories. They are a fascinating genre that can evoke strong emotions and create a sense of suspense and fear. Do you have any specific horror stories in mind that you would like to discuss?
User: Sara, Tell me a one paragraph story.
Sara: Here's a chilling tale for you: Margaret had always been proud of her antique music box, inherited from her grandmother, until she noticed it playing its haunting melody every night at exactly 3:17 AM—despite never winding it. The delicate ballerina inside would spin slowly in the darkness, casting eerie shadows on her bedroom wall while the tinkling notes of "Für Elise" drifted through the silence. When she finally worked up the courage to examine it closely one night, she discovered a small inscription inside the lid that made her blood run cold: "For Margaret, who dances with me in the dark - Love, Grandmother," dated just three days ago—two years after he